# 01 — Data Understanding (Phases 1.5, 1.6, 2)

Profile the accepted & rejected files, then build the leakage-safe feature set.

**Before running:** click **Select Kernel** (top-right) -> **Python Environments** -> **`.venv`**, then run cells top to bottom (▶▶ *Run All*).

In [ ]:
import pandas as pd
from pathlib import Path

# Robust path: works whether the working dir is the repo root OR notebooks/
DATA_DIR = next(p for p in [Path('data'), Path('../data')] if p.exists())
ACCEPTED = DATA_DIR / 'accepted_2007_to_2018Q4.csv.gz'
REJECTED = DATA_DIR / 'rejected_2007_to_2018Q4.csv.gz'
print('Data dir:', DATA_DIR.resolve())

# 2.26M x 151 ~ 3-4 GB in RAM. low_memory=False reads each column in one pass
# (also silences the 'mixed types in id' warning). Takes ~30-60s.
acc = pd.read_csv(ACCEPTED, low_memory=False)
print('accepted shape:', acc.shape)

## Phase 1.5 - Profile the accepted file
Shape, dtypes, memory, the target column, and the junk rows.

In [ ]:
# dtypes + TRUE memory footprint ('deep' counts string memory properly)
acc.info(memory_usage='deep')

In [ ]:
acc.head()

In [ ]:
# Raw target source -> becomes a binary default flag in Phase 3
acc['loan_status'].value_counts(dropna=False)

In [ ]:
# The mixed-type 'id' warning hinted at non-loan summary rows.
# Real loans always have a loan_amnt; junk rows don't.
print('rows with no loan_amnt (junk):', acc['loan_amnt'].isna().sum())
acc[acc['loan_amnt'].isna()].tail()

## Phase 1.6 - Inspect the rejected file
27.6M rows, only 9 columns. Sample to inspect cheaply. These 9 columns cap what reject inference (Phase 10) can ever use.

In [ ]:
# Sample 200k rows -- enough to understand structure without loading 27.6M
rej = pd.read_csv(REJECTED, nrows=200_000)
print('rejected sample shape:', rej.shape)
rej.head()

In [ ]:
rej.info()
rej.describe(include='all')

In [ ]:
# Policy Code should be constant (useless); Risk_Score is the FICO-like field
print(rej['Policy Code'].value_counts(dropna=False))
print()
print('Risk_Score summary:')
print(rej['Risk_Score'].describe())

## Phase 2 - Apply the leakage exclusion list
Define the banned columns, verify they exist, then derive & save the allowed feature set.

In [ ]:
# Translated from reports/leakage_exclusion_list.md

TARGET = 'loan_status'

TIME_KEY = 'issue_d'  # used for the out-of-time split, NOT a feature

# --- Banned: recorded AFTER origination (these encode the outcome) ---

LEAK_PAYMENT = [
    'out_prncp',
    'out_prncp_inv',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_prncp',
    'total_rec_int',
    'total_rec_late_fee',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d',
    'last_pymnt_amnt',
    'next_pymnt_d',
    'last_credit_pull_d',
    'last_fico_range_high',
    'last_fico_range_low',
    'pymnt_plan',
]

LEAK_HARDSHIP = [
    'hardship_flag',
    'hardship_type',
    'hardship_reason',
    'hardship_status',
    'deferral_term',
    'hardship_amount',
    'hardship_start_date',
    'hardship_end_date',
    'payment_plan_start_date',
    'hardship_length',
    'hardship_dpd',
    'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',
]

LEAK_SETTLEMENT = [
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term',
]

# --- Excluded by judgment: Lending Club's own risk pricing ---

LC_PRICING = [
    'grade',
    'sub_grade',
    'int_rate',
    'installment',
]

# --- Excluded: identifiers / free text / constants / redundant ---

DROP_OTHER = [
    'id',
    'member_id',
    'url',
    'policy_code',
    'funded_amnt',
    'funded_amnt_inv',
    'emp_title',
    'title',
    'desc',
]

EXCLUDE = set(LEAK_PAYMENT + LEAK_HARDSHIP + LEAK_SETTLEMENT
              + LC_PRICING + DROP_OTHER + [TARGET, TIME_KEY])
print('total columns excluded:', len(EXCLUDE))

In [ ]:
# Safety check: every banned name should actually exist in the file
missing = [c for c in EXCLUDE if c not in acc.columns]
print('listed-but-not-found (should be empty):', missing)

# Allowed features = everything else
ALLOWED = [c for c in acc.columns if c not in EXCLUDE]
print('allowed feature count:', len(ALLOWED))
ALLOWED

In [ ]:
# Persist the allowed feature list so later phases reuse the exact same set
out = DATA_DIR.parent / 'reports' / 'allowed_features.txt'
pd.Series(ALLOWED).to_csv(out, index=False, header=False)
print('saved', len(ALLOWED), 'features to', out)